# Bài tập 09 - Mạng nơ-ron nhiều lớp (MLP)

Sinh viên: **Lê Quyết Tiến** - MSSV: **2474802010386**.

Notebook chạy từ thư mục gốc `09. MLP`. Trên Colab, hãy mount Google Drive, `%cd` đến thư mục này rồi chọn **Runtime > Run all**. Giữ nguyên các tệp CSV trong `data/`.


## Bài 1


In [1]:
# -*- coding: utf-8 -*-
"""Bài 1: tự dự đoán bằng trọng số MLP đã học."""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier


def relu(z):
    return np.maximum(0, z)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


df = pd.read_csv("data/xor.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
mlp = MLPClassifier(
    hidden_layer_sizes=(8,), solver="lbfgs", max_iter=5000, random_state=0
).fit(X_tr, y_tr)

X5 = X_te.iloc[:5].to_numpy()
W1, W2 = mlp.coefs_
b1, b2 = mlp.intercepts_
H = relu(X5 @ W1 + b1)
xac_suat_tu_tinh = sigmoid(H @ W2 + b2).ravel()
xac_suat_thu_vien = mlp.predict_proba(X_te.iloc[:5])[:, 1]

print(" i   tu tinh        predict_proba   chenh lech")
for i, (a, b) in enumerate(zip(xac_suat_tu_tinh, xac_suat_thu_vien), 1):
    print(f" {i}   {a:.10f}    {b:.10f}      {abs(a-b):.2e}")
print("\nSai khac lon nhat:", np.max(np.abs(xac_suat_tu_tinh - xac_suat_thu_vien)))
print("\nNhan xet:")
print("- Lop an tinh X @ W1 + b1 roi dua qua ReLU; lop ra tinh H @ W2 + b2 roi qua sigmoid.")
print("- Ket qua tu tinh trung voi predict_proba den sai so may, chung to lan truyen tien da dung.")
print("- coefs_ chua ma tran trong so giua cac lop; intercepts_ chua he so chan cua tung lop.")


 i   tu tinh        predict_proba   chenh lech
 1   0.0175579854    0.0175579854      0.00e+00
 2   0.9994348320    0.9994348320      0.00e+00
 3   0.7943310566    0.7943310566      0.00e+00
 4   0.1344008896    0.1344008896      2.78e-17
 5   0.9578057181    0.9578057181      0.00e+00

Sai khac lon nhat: 2.7755575615628914e-17

Nhan xet:
- Lop an tinh X @ W1 + b1 roi dua qua ReLU; lop ra tinh H @ W2 + b2 roi qua sigmoid.
- Ket qua tu tinh trung voi predict_proba den sai so may, chung to lan truyen tien da dung.
- coefs_ chua ma tran trong so giua cac lop; intercepts_ chua he so chan cua tung lop.


## Bài 2


In [2]:
# -*- coding: utf-8 -*-
"""Bài 2: kiểm chứng một mạng khác cũng tính đúng XOR."""

import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
h1 = np.maximum(0, X[:, 0] - X[:, 1])
h2 = np.maximum(0, X[:, 1] - X[:, 0])
y = h1 + h2

print("x1 x2  h1  h2  y")
for x, a, b, c in zip(X, h1, h2, y):
    print(f" {x[0]:.0f}  {x[1]:.0f}   {a:.0f}   {b:.0f}  {c:.0f}")

print("\nNhan xet va giai thich:")
print("- Neu x1=x2 thi ca x1-x2 va x2-x1 deu bang 0, nen h1=h2=0 va y=0.")
print("- Neu x1 khac x2, mot trong hai hieu bang 1 con hieu kia bang -1.")
print("  ReLU giu so 1 va bien so -1 thanh 0, nen h1+h2=1.")
print("- Vi vay y bang 1 khi hai dau vao khac nhau va bang 0 khi chung giong nhau: dung la XOR.")


x1 x2  h1  h2  y
 0  0   0   0  0
 0  1   0   1  1
 1  0   1   0  1
 1  1   0   0  0

Nhan xet va giai thich:
- Neu x1=x2 thi ca x1-x2 va x2-x1 deu bang 0, nen h1=h2=0 va y=0.
- Neu x1 khac x2, mot trong hai hieu bang 1 con hieu kia bang -1.
  ReLU giu so 1 va bien so -1 thanh 0, nen h1+h2=1.
- Vi vay y bang 1 khi hai dau vao khac nhau va bang 0 khi chung giong nhau: dung la XOR.


## Bài 3


In [3]:
# -*- coding: utf-8 -*-
"""Bài 3: so sánh các hàm kích hoạt trên bộ hai vầng trăng."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

ket_qua = []
for kich_hoat in ["logistic", "tanh", "relu"]:
    mh = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(100, 100), activation=kich_hoat, alpha=1.0,
            solver="lbfgs", max_iter=20000, random_state=0
        ),
    ).fit(X_tr, y_tr)
    ket_qua.append((
        kich_hoat,
        accuracy_score(y_tr, mh.predict(X_tr)),
        accuracy_score(y_te, mh.predict(X_te)),
    ))

print("ham kich hoat   tap hoc   tap kiem tra")
for ten, hoc, kt in ket_qua:
    print(f"{ten:13s}   {hoc:.4f}       {kt:.4f}")
diem_tot = max(dong[2] for dong in ket_qua)
ten_tot = [dong[0] for dong in ket_qua if abs(dong[2] - diem_tot) < 1e-12]
print("\nNhan xet:")
print(f"- Tren lan chia du lieu nay, {' va '.join(ten_tot)} cung dat accuracy kiem tra cao nhat: {diem_tot:.4f}.")
print("- Ham kich hoat tao dang phi tuyen khac nhau; ket qua con phu thuoc du lieu va dieu chuan.")
print("- Nen chon bang kiem dinh cheo thay vi ket luan mot ham luon tot nhat.")


ham kich hoat   tap hoc   tap kiem tra
logistic        0.8400       0.8533
tanh            0.8711       0.9067
relu            0.8800       0.9067

Nhan xet:
- Tren lan chia du lieu nay, tanh va relu cung dat accuracy kiem tra cao nhat: 0.9067.
- Ham kich hoat tao dang phi tuyen khac nhau; ket qua con phu thuoc du lieu va dieu chuan.
- Nen chon bang kiem dinh cheo thay vi ket luan mot ham luon tot nhat.


## Bài 4


In [4]:
# -*- coding: utf-8 -*-
"""Bài 4: ảnh hưởng của tốc độ học trên bộ chữ số."""

from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

fig, ax = plt.subplots(figsize=(8, 5))
ket_qua = []
for toc_do in [0.0001, 0.001, 0.01, 0.1]:
    mh = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(100,), learning_rate_init=toc_do,
            max_iter=1000, random_state=0
        ),
    ).fit(X_tr, y_tr)
    mlp = mh[-1]
    acc = accuracy_score(y_te, mh.predict(X_te))
    ket_qua.append((toc_do, mlp.n_iter_, mlp.loss_curve_[-1], acc))
    ax.plot(mlp.loss_curve_, label=f"learning_rate={toc_do:g}")

ax.set_xlabel("Vong hoc")
ax.set_ylabel("Loss")
ax.set_title("Anh huong cua learning_rate_init")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/toc_do_hoc.png", dpi=160, bbox_inches="tight")
if "agg" not in plt.get_backend().lower():
    plt.show()
else:
    plt.close(fig)

print("toc do hoc   so vong   loss cuoi   accuracy")
for toc_do, so_vong, loss, acc in ket_qua:
    print(f"{toc_do:10g}   {so_vong:7d}   {loss:9.5f}   {acc:.4f}")
print("\nNhan xet:")
print("- Toc do qua nho lam loss giam cham, can nhieu vong va co the chua den diem tot.")
print("- Toc do qua lon co the vuot qua diem toi uu, loss dao dong hoac hoi tu den nghiem kem.")
print("- Toc do vua phai giup loss giam nhanh, on dinh va accuracy kiem tra cao.")
print("- Da luu hinh tai figures/toc_do_hoc.png.")


toc do hoc   so vong   loss cuoi   accuracy
    0.0001       582     0.01893   0.9889
     0.001       144     0.00521   0.9844
      0.01        38     0.00108   0.9778
       0.1        30     0.12037   0.9511

Nhan xet:
- Toc do qua nho lam loss giam cham, can nhieu vong va co the chua den diem tot.
- Toc do qua lon co the vuot qua diem toi uu, loss dao dong hoac hoi tu den nghiem kem.
- Toc do vua phai giup loss giam nhanh, on dinh va accuracy kiem tra cao.
- Da luu hinh tai figures/toc_do_hoc.png.


## Bài 5


In [5]:
# -*- coding: utf-8 -*-
"""Bài 5: dò lưới cấu trúc lớp ẩn và alpha cho MLP."""

from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

pipe = make_pipeline(
    StandardScaler(),
    MLPClassifier(max_iter=500, random_state=0),
)
luoi = {
    "mlpclassifier__hidden_layer_sizes": [(10,), (50,), (100,), (50, 50)],
    "mlpclassifier__alpha": [0.0001, 0.01, 1.0],
}
tim = GridSearchCV(pipe, luoi, cv=3, n_jobs=-1)
tim.fit(X_tr, y_tr)
acc_mlp = accuracy_score(y_te, tim.predict(X_te))
svm = SVC().fit(X_tr, y_tr)
acc_svm = accuracy_score(y_te, svm.predict(X_te))

print("Cap tham so tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo tot nhat: {tim.best_score_:.4f}")
print(f"Accuracy MLP tren tap kiem tra: {acc_mlp:.4f}")
print(f"Accuracy SVM kernel RBF:        {acc_svm:.4f}")
print("\nNhan xet:")
print(f"- Cau hinh tot nhat theo kiem dinh cheo la {tim.best_params_}.")
print(f"- MLP {'vuot' if acc_mlp > acc_svm else 'khong vuot'} SVM tren tap kiem tra nay.")
print("- Tap kiem tra chi duoc dung sau khi chon tham so; khong dung no de chon mo hinh.")


Cap tham so tot nhat: {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (100,)}
Diem kiem dinh cheo tot nhat: 0.9777
Accuracy MLP tren tap kiem tra: 0.9844
Accuracy SVM kernel RBF:        0.9911

Nhan xet:
- Cau hinh tot nhat theo kiem dinh cheo la {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (100,)}.
- MLP khong vuot SVM tren tap kiem tra nay.
- Tap kiem tra chi duoc dung sau khi chon tham so; khong dung no de chon mo hinh.


C:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


## Bài 6


In [6]:
# -*- coding: utf-8 -*-
"""Bài 6: mạng nơ-ron cho hồi quy giá nhà."""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17
)

tuyen_tinh = LinearRegression().fit(X_tr, y_tr)
mlp = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(20,), max_iter=5000, random_state=0),
).fit(X_tr, y_tr)

mae_lr = mean_absolute_error(y_te, tuyen_tinh.predict(X_te))
mae_mlp = mean_absolute_error(y_te, mlp.predict(X_te))
moi = pd.DataFrame({"dien_tich": [250.0]})
gia_lr = tuyen_tinh.predict(moi)[0]
gia_mlp = mlp.predict(moi)[0]
print(f"MAE hoi quy tuyen tinh: {mae_lr:.4f}")
print(f"MAE MLPRegressor:       {mae_mlp:.4f}")
print(f"Gia du doan can 250 m2 - tuyen tinh: {gia_lr:.4f}")
print(f"Gia du doan can 250 m2 - MLP:        {gia_mlp:.4f}")
print(f"Khoang dien tich du lieu hoc: {X_tr['dien_tich'].min():.1f} den {X_tr['dien_tich'].max():.1f} m2")

print("\nNhan xet:")
print("- Can 250 m2 nam ngoai khoang dien tich hoc, nen day la bai toan ngoai suy.")
print("- Hoi quy tuyen tinh tiep tuc duong thang ra ngoai mien hoc.")
print("- MLP hoc quan he phi tuyen trong mien da thay, nhung khong bi rang buoc phai tiep tuc theo")
print("  xu huong duong thang; vi vay ngoai suy thuong kem on dinh hon hoi quy tuyen tinh.")


MAE hoi quy tuyen tinh: 0.3123
MAE MLPRegressor:       0.5050
Gia du doan can 250 m2 - tuyen tinh: 20.0672
Gia du doan can 250 m2 - MLP:        28.6338
Khoang dien tich du lieu hoc: 37.6 den 115.9 m2

Nhan xet:
- Can 250 m2 nam ngoai khoang dien tich hoc, nen day la bai toan ngoai suy.
- Hoi quy tuyen tinh tiep tuc duong thang ra ngoai mien hoc.
- MLP hoc quan he phi tuyen trong mien da thay, nhung khong bi rang buoc phai tiep tuc theo
  xu huong duong thang; vi vay ngoai suy thuong kem on dinh hon hoi quy tuyen tinh.
